# Uplift-таргетинг и сценарная экономика

Notebook визуализирует только рассчитанные артефакты. Денежные показатели являются сценарными и зависят от переданных параметров.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

ARTIFACT_DIR = Path('../data/processed/business_optimization')
paths = {
    'customers': ARTIFACT_DIR / 'customer_targeting.parquet',
    'strategies': ARTIFACT_DIR / 'business_scenarios.parquet',
    'thresholds': ARTIFACT_DIR / 'threshold_optimization.parquet',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Нет бизнес-артефактов. Выполните python -m src.business.optimization ' + str(missing)
    )

## Сравнение стратегий

In [ ]:
strategies = pl.read_parquet(paths['strategies']).to_pandas()
display(strategies)
sns.barplot(data=strategies, x='strategy', y='incremental_profit')
plt.axhline(0, color='black', linewidth=1)
plt.title('Сценарная incremental profit')
plt.show()

## Чувствительность к порогу

In [ ]:
thresholds = pl.read_parquet(paths['thresholds']).sort('customers_targeted').to_pandas()
sns.lineplot(data=thresholds, x='target_share', y='incremental_profit')
best = thresholds.loc[thresholds['incremental_profit'].idxmax()]
plt.scatter([best['target_share']], [best['incremental_profit']], color='red', label='Лучший сценарий')
plt.axhline(0, color='black', linewidth=1)
plt.title('Прибыль в зависимости от размера аудитории')
plt.legend()
plt.show()

## Модельные типы клиентов

In [ ]:
customers = pl.read_parquet(paths['customers'])
type_summary = (
    customers.group_by('modeled_customer_type')
    .agg(pl.len().alias('customers'), pl.col('is_targeted').mean().alias('target_share'))
    .sort('customers', descending=True)
).to_pandas()
display(type_summary)
sns.barplot(data=type_summary, x='customers', y='modeled_customer_type')
plt.title('Клиенты по модельным типам')
plt.show()